# LC 10 — Regular Expression Matching
**Day-63 | Hard | Dynamic Programming**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Build a DP table where
<code>dp[i][j]</code> means "does <code>s[:i]</code> match
<code>p[:j]</code>?". The tricky part is <code>'*'</code>:
it either matches zero occurrences of the preceding char
(skip both: <code>dp[i][j-2]</code>) or one-or-more
(if chars match: <code>dp[i-1][j]</code>).
</div>

## Official Problem Statement

Given an input string `s` and a pattern `p`, implement regular
expression matching with support for `'.'` and `'*'` where:
- `'.'` Matches any single character.
- `'*'` Matches zero or more of the preceding element.

The matching must cover the **entire** input string (not partial).

**Example 1:**
```
Input:  s = "aa", p = "a"
Output: False   (pattern must match entire string)
```
**Example 2:**
```
Input:  s = "aa", p = "a*"
Output: True    ('a*' = zero or more 'a')
```
**Example 3:**
```
Input:  s = "ab", p = ".*"
Output: True    ('.*' = zero or more of any char)
```

**Constraints:**
- `1 <= s.length <= 20`
- `1 <= p.length <= 30`
- `s` contains only lowercase English letters.
- `p` contains only lowercase English letters, `'.'`, and `'*'`.
- It is guaranteed for each `'*'` there will be a valid
  preceding character to match.

## What This Is Actually Asking

We are implementing a simplified regex engine that handles
only two special characters: `'.'` (any char) and `'*'`
(repeat preceding char zero or more times).
The hard part is `'*'` — it creates branching choices:
use it zero times (skip it) or consume one more character.
Dynamic programming solves this by building up correctness
from empty strings, so every subproblem is already solved
when we need it.
The answer is whether the full string and full pattern match,
i.e., `dp[len(s)][len(p)]`.

## Walk Through an Example by Hand

`s = "aab"`, `p = "c*a*b"`

```
    ""  c  *  a  *  b
""   T  F  T  F  T  F
a    F  F  F  T  T  F
a    F  F  F  F  T  F
b    F  F  F  F  F  T  <-- answer

Key transitions:
dp[0][2]: p[1]='*' -> dp[0][0] (zero c's) = True
dp[0][4]: p[3]='*' -> dp[0][2] (zero a's) = True

dp[1][3]: p[2]='a', s[0]='a' -> match! dp[0][2] = True
dp[1][4]: p[3]='*' ->
  zero:  dp[1][2] = False
  one+:  match('a','a')=True AND dp[0][4]=True -> True

dp[2][4]: p[3]='*' ->
  zero:  dp[2][2] = False
  one+:  match('a','a')=True AND dp[1][4]=True -> True

dp[3][5]: p[4]='b', s[2]='b' -> match! dp[2][4] = True
```
Answer: `dp[3][5] = True`

## The Picture

```
DP table for s="aab", p="c*a*b"

Indices:   j=0  1  2  3  4  5
Pattern:    ""  c  *  a  *  b
           +--+--+--+--+--+--+
i=0  ""    | T| F| T| F| T| F|
           +--+--+--+--+--+--+
i=1   a    | F| F| F| T| T| F|
           +--+--+--+--+--+--+
i=2   a    | F| F| F| F| T| F|
           +--+--+--+--+--+--+
i=3   b    | F| F| F| F| F| T|  <-- answer here
           +--+--+--+--+--+--+

Helper: match(s_char, p_char)
  = (p_char == '.' OR p_char == s_char)

Rules:
  if p[j-1] == '*':
    dp[i][j] = dp[i][j-2]           # zero occurrences
             OR (
               match(s[i-1], p[j-2])
               AND dp[i-1][j]        # one or more
             )
  else:
    dp[i][j] = match(s[i-1], p[j-1])
               AND dp[i-1][j-1]
```

## When To Use This Pattern

- When a problem involves **matching two sequences** with
  special rules (wildcards, gaps), think 2D DP.
- When a character in one sequence can match **zero or more**
  characters in another, think DP with a `j-2` skip.
- When choices branch (use `'*'` or skip it), and those
  choices overlap, think DP over recursion with memoization.
- When `'.'` can match anything and `'*'` repeats, this
  exact dp[i][j] table is the canonical solution.
- When asked about string matching, parsing, or pattern
  validation in interviews, think 2D DP.

## The Approach

Create a `(m+1) x (n+1)` boolean DP table where rows represent
prefixes of `s` and columns represent prefixes of `p`.
Initialize `dp[0][0] = True` (empty matches empty), then handle
patterns like `a*b*c*` that can match the empty string by
seeding the first row.
Fill the table row by row: if the current pattern char is `'*'`,
check the zero-use case (`dp[i][j-2]`) and the one-or-more
case (`match AND dp[i-1][j]`); otherwise just check a direct
character match combined with the diagonal cell.
Return `dp[m][n]`.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    cases = [
        # (s, p, expected, label)
        ("aa",  "a",    False, "no full match"),
        ("aa",  "a*",   True,  "star repeats"),
        ("ab",  ".*",   True,  "dot-star all"),
        ("aab", "c*a*b",True,  "classic"),
        ("mississippi", "mis*is*p*.",
         False, "tricky backtrack"),
        ("",    "a*",   True,  "empty string star"),
        ("",    "",     True,  "both empty"),
        ("a",   ".",    True,  "dot match"),
        ("a",   "b",    False, "no match"),
        ("abc", "a.c",  True,  "dot middle"),
        ("aaa", "a*a",  True,  "star then literal"),
    ]
    passed = 0
    for s, p, expected, label in cases:
        result = func(s, p)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status} [{label}]: "
                f"s={s!r} p={p!r} "
                f"got {result}, expected {expected}"
            )
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")

In [ ]:
def isMatch(s: str, p: str) -> bool:
    """
    Regex matching with '.' and '*' using 2D DP.

    Strategy:
        dp[i][j] = does s[:i] match p[:j]?
        Base: dp[0][0] = True
        First row: dp[0][j] = True if p[j-1]=='*'
                               and dp[0][j-2]
        Fill:
          if p[j-1] == '*':
            dp[i][j] = dp[i][j-2]   # zero use
              or (match(s[i-1],p[j-2]) and dp[i-1][j])
          else:
            dp[i][j] = match(s[i-1],p[j-1])
                        and dp[i-1][j-1]
        match(a,b) = b=='.' or a==b

    Args:
        s: Input string (lowercase letters only)
        p: Pattern (lowercase letters, '.', '*')

    Returns:
        True if p matches entire string s

    Time:  O(m * n) — fill (m+1)x(n+1) table
    Space: O(m * n) — DP table storage
    """
    # Debug: show inputs
    print(f"[debug] s={s!r}  p={p!r}")

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(isMatch)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Recursive (naive) | Exponential | O(m+n) stack | Re-solves subproblems |
| Recursion + memo | O(m * n) | O(m * n) | Top-down DP |
| **Bottom-up DP** | **O(m * n)** | **O(m * n)** | **Iterative, clear** |
| DP (1-row opt.) | O(m * n) | O(n) | Space-optimized |

Where m = len(s), n = len(p).

## Real World Connection

Regex matching is the foundation of every log parser and
data validation pipeline — at Citi, trade confirmation
messages are validated against patterns exactly like this.
AWS Glue and Athena use regex-based schema inference when
cataloging semi-structured data in S3, matching field
patterns against expected formats.
The DP approach here teaches a broader lesson: when a
greedy or recursive solution has overlapping subproblems
("should I use this `*` or not?"), memoize the states.
This same `dp[i][j]` pattern underpins edit distance,
sequence alignment in bioinformatics, and fuzzy string
matching in search engines.
Understanding it deeply makes other 2D DP problems feel
like variations on a theme.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra